# 1.1 Prompting — Talking to LLMs Effectively

## What you will learn in this notebook

- How **basic prompting** works and why wording matters
- How a **system prompt** shapes the agent's personality and behaviour
- How **few-shot examples** teach the model by demonstration
- How **structured prompts** control the *shape* of the reply
- How **structured output** forces the model to return machine-readable data (Pydantic)

---

### Key vocabulary before we start

| Term | Plain English |
|---|---|
| **Prompt** | Any text you send to a language model. It sets the context for the reply. |
| **System Prompt** | A special instruction block that runs *before* the user's message. It defines the model's role, persona, rules, and output format. The user never sees it. |
| **Few-shot examples** | Sample Q&A pairs included in the prompt that show the model *exactly* the style/format you want — learning by example, not by explanation. |
| **Zero-shot** | No examples provided — the model relies on its training alone. |
| **Structured prompt** | A system prompt that tells the model to stick to a specific *text* template (Name:, Location:, etc.). |
| **Structured output** | Going further — using Pydantic to force the model to return a typed Python *object* rather than free text. Much easier to use in code. |
| **Pydantic** | A Python library for data validation. We define a class that describes the exact fields we want, and the model fills them in. |

---

### The prompting ladder (from weakest to strongest control)

```
Basic prompt          → model does what it thinks is best
System prompt         → model adopts a persona / role
Few-shot examples     → model copies your demonstrated style
Structured prompt     → model fills a text template
Structured output     → model returns a typed Python object  ← most reliable
```

We will climb this ladder one step at a time.

---
## Section 1 — Basic Prompting

The simplest possible interaction: send a question, get a reply.

At this level, the model uses **only its training knowledge** to answer. There are no special instructions, no persona, no format constraints. The model decides everything — tone, length, style — on its own.

This is useful as a **baseline** to see the model's default behaviour before we start tuning it.

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Reads API keys from the .env file in the project root and injects
# them into os.environ. LangChain picks them up automatically.
# Your .env should contain at minimum: OPENAI_API_KEY=sk-...
# Never hardcode keys in notebooks — they end up in Git history!

from dotenv import load_dotenv

load_dotenv()

True

In [7]:
# ============================================================
# CELL 2: Basic Prompt — No System Instructions
# ============================================================
# This is the simplest possible agent call:
#   1. Create an agent backed by gpt-5-nano (no system prompt)
#   2. Wrap our question in a HumanMessage
#   3. Invoke the agent and read the reply
#
# HumanMessage tells LangChain this text came from the *user* role.
# This matters because LLMs are trained on structured chat logs where
# every message has a role: system / human / assistant / tool.
#
# response['messages'] is a list of all messages in the conversation:
#   index [0] → HumanMessage  (our question)
#   index [1] → AIMessage     (the model's reply)  ← we print this
#
# Expected output: a factual answer saying the Moon has no capital.
# The model with no persona stays grounded in reality.

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
agent = create_agent(model=llm)

# Store the question in a variable — we'll reuse it across cells
question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

# [1] is the second message = the AI's reply to our question

print(response['messages'][1].content)

The Moon does not have a capital city. The Moon is a natural satellite that orbits the Earth, and it does not have a human settlement or a government that would establish a capital city. While there have been several manned missions to the Moon in the past, there are no permanent human settlements or cities on the Moon.


---
## Section 2 — System Prompts

### What is a system prompt?

A **system prompt** is a hidden instruction block that is injected *before* the conversation starts. The user never sees it, but the model reads it first and uses it to understand:

- **Who it is** — its persona or role
- **What it should do** — its task or goal
- **How it should behave** — tone, style, rules, constraints

Think of it as briefing an employee before a customer call: "You are a friendly support agent for XYZ bank. Never discuss competitor products. Always end with 'Is there anything else I can help you with?'"

### How it changes the model

```
Without system prompt:  "The Moon has no capital — it's not a country."
With sci-fi system prompt: "The gleaming city of Selenopolis crowns the lunar highlands..."
```

Same question, completely different answer — because the *role* changed.

In [9]:
# ============================================================
# CELL 3: Adding a System Prompt — Give the Model a Persona
# ============================================================
# The system_prompt parameter in create_agent is injected as a
# SystemMessage at the very beginning of every conversation.
#
# Key parts of this system prompt:
#   "You are a science fiction writer"  → defines the ROLE
#   "create a capital city"             → defines the TASK
#   "at the users request"              → defines the TRIGGER
#
# With this persona, the model is now "allowed" to invent fictional
# cities instead of sticking to factual answers. The same question
# that previously got a factual refusal now gets a creative response.
#
# 💡 Best practice: system prompts should be specific and unambiguous.
#    Vague prompts like "be helpful" give the model too much freedom
#    and produce inconsistent results.

system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=system_prompt   # Injected as SystemMessage before every conversation
)

# We reuse the same `question` variable from Cell 2
response = scifi_agent.invoke(
    {"messages": [question]}
)

# [1] is still the AI's first reply
print(response['messages'][1].content)

While there isn't a real capital city on the moon (yet!), I can create a fictional one for you.

Let's call the capital city of the moon "Lunaria". Located in the heart of the moon's near side, Lunaria is a thriving metropolis that serves as the hub of lunar politics, culture, and innovation.

Geography and Architecture:
Lunaria is situated in the vast, dark plain of the Mare Imbrium, surrounded by a ring of majestic mountains that stretch towards the sky like giant sentinels. The city's architecture is a blend of futuristic and sustainable design, with towering domes and spires that seem to defy gravity. The buildings are made from a unique lunar material called "Moonstone," a durable and lightweight metal alloy that is both strong and flexible.

The city's central square, known as the "Lunar Forum," is a grand, open space that hosts various cultural and scientific events throughout the year. At its center stands a magnificent statue of the moon's patron deity, Selene, goddess of the 

---
## Section 3 — Few-Shot Examples

### What are few-shot examples?

**Few-shot prompting** means including 2–5 worked examples directly inside your system prompt to show the model *exactly* the style, format, and tone you want.

It exploits a key property of LLMs: they are pattern-completion machines. If you show them a consistent pattern, they will continue it.

| Approach | Examples given | Use when |
|---|---|---|
| **Zero-shot** | 0 | You trust the model to infer the format |
| **One-shot** | 1 | You have one strong example |
| **Few-shot** | 2–5 | You want consistent style/format |
| **Many-shot** | 10+ | Fine-tuning territory — better to train at this point |

### Why does this work?

The model sees:
```
User: What is the capital of Mars?     → short question format
Scifi Writer: Marsialis               → short, invented single-word answer

User: What is the capital of Venus?
Scifi Writer: Venusovia               → same pattern

User: What's the capital of the moon? → new question, same format
Scifi Writer: ???                     → model continues the pattern
```

Without few-shot: the model might write a 3-paragraph creative story.  
With few-shot: the model gives a concise invented city name — matching the pattern.

In [10]:
# ============================================================
# CELL 4: Few-Shot Examples Inside the System Prompt
# ============================================================
# We embed two Q&A examples directly in the system prompt.
# This teaches the model:
#   1. The answer should be a SHORT invented city name (not a paragraph)
#   2. The naming convention: planet name + Latin/fantasy suffix
#   3. No preamble, no explanation — just the name
#
# Notice the format uses "User:" and "Scifi Writer:" labels.
# These are plain text labels (not LangChain message objects).
# They work because the model has seen millions of dialogue
# transcripts in this format during training.
#
# Triple-quoted strings ("""...""") let us write multi-line
# system prompts cleanly without manual \n characters.
#
# 💡 Tip: always check that your examples are consistent.
#    Contradictory examples confuse the model and produce
#    unpredictable results.

system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

# Compare this output to Cell 3 — the few-shot examples should
# make the answer shorter and follow the Marsialis / Venusovia naming style
print(response['messages'][1].content)

The capital of the moon is Lunaria City, a sprawling metropolis built into the cratered surface of the lunar regolith. Located in the heart of the Moon's near side, Lunaria City is a hub of intergalactic commerce, innovation, and culture, with towering domes and spires that shine like beacons in the stark, airless beauty of the lunar landscape.


---
## Section 4 — Structured Prompts

### What is a structured prompt?

A structured prompt tells the model to **fill in a text template** — like a form. Instead of free-form prose, the model must follow a specific field-by-field layout.

```
Name: Selenopolis
Location: Mare Tranquillitatis, equatorial region
Vibe: Sleek, ethereal, austere
Economy: Helium-3 mining, lunar tourism, zero-gravity research
```

### When to use structured prompts vs structured output

| | Structured Prompt | Structured Output (Pydantic) |
|---|---|---|
| **Returns** | Formatted text string | Python object with typed fields |
| **Reliability** | Model *might* deviate | Guaranteed schema — model can't skip fields |
| **Use in code** | Need to parse the string manually | Access via `response.name`, `response.location` |
| **Best for** | Human-readable display | Feeding into downstream code / databases |

> 💡 Structured prompts are a good stepping stone. But in production, always prefer **structured output** (next section) — it is far more robust.

In [11]:
# ============================================================
# CELL 5: Structured Prompt — Text Template
# ============================================================
# We add a template to the system prompt that defines exactly
# which fields we want and what each one means.
#
# How it works:
#   The model sees the template and understands it must produce
#   output in exactly that format — one field per line.
#
# Advantages:
#   - Output is human-readable and consistently formatted
#   - Easy to scan visually
#
# Disadvantages:
#   - The model may occasionally deviate (add extra text, rename fields)
#   - To use the data in code you must parse the string manually
#     (e.g. split on "\n", then split on ":") — fragile and error-prone
#
# This is why in the next section we move to Pydantic structured output.

system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

# Output should now have Name / Location / Vibe / Economy sections.
# Notice it is still a plain string — we cannot do response.name yet.
print(response['messages'][1].content)

Name: Lunaria

Location: Southern hemisphere of the Moon, within the Shackleton crater

Vibe: Futuristic Oasis

Economy: The main industries in Lunaria are Helium-3 mining, Space Tourism, and Advanced Manufacturing, with a growing focus on Lunar-based Renewable Energy and Interplanetary Trade.


---
## Section 5 — Structured Output with Pydantic

### The problem with structured prompts

Even with a clear template, the model might reply:
```
Sure! Here's a city for you:
Name: Selenopolis
Location: ...
```
That preamble ("Sure! Here's a city for you:") breaks any string parser you write. Or the model might add a "Motto:" field you didn't ask for.

### The solution: Pydantic + `response_format`

**Pydantic** is Python's most popular data validation library. We define a class that describes the exact fields we want — their names and types. LangChain then instructs the model to return data that matches this schema *exactly*, with no extra text, no missing fields.

```python
class CapitalInfo(BaseModel):
    name: str       # must be a string
    location: str   # must be a string
    vibe: str       # must be a string
    economy: str    # must be a string
```

The model no longer returns a message string. It returns a **validated Python object** — as if someone parsed the text for you and put it into a neat dataclass.

### How it works under the hood

1. LangChain converts your Pydantic class into a **JSON Schema**
2. It sends that schema to the model as part of the request
3. The model uses **function/tool calling** internally to fill the schema fields
4. LangChain validates the output against your class and raises an error if it doesn't match
5. You get back a clean Python object — no parsing needed

In [12]:
# ============================================================
# CELL 6: Structured Output — Define the Schema with Pydantic
# ============================================================
# Step 1: Define what fields you want using a Pydantic BaseModel.
#
#   class CapitalInfo(BaseModel):
#       name: str      → field called 'name', must be a string
#       location: str  → field called 'location', must be a string
#       vibe: str      → field called 'vibe', must be a string
#       economy: str   → field called 'economy', must be a string
#
# You can also use:
#   int, float, bool, list[str], Optional[str], etc.
#   Field(description="...") to give the model hints about each field
#
# Step 2: Pass the class to create_agent via response_format=CapitalInfo
#   This tells LangChain to enforce this schema on every response.
#
# Step 3: After .invoke(), the result is in response["structured_response"]
#   This key exists ONLY when response_format is set.
#   It holds a CapitalInfo OBJECT, not a string.

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

# Define the output schema — this is the "contract" between us and the model
class CapitalInfo(BaseModel):
    name: str       # Name of the invented city
    location: str   # Where on the celestial body it sits
    vibe: str       # 2-3 word atmosphere description
    economy: str    # Main industries

agent = create_agent(
    model='groq:llama-3.3-70b-versatile',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo   # ← enforces our schema on every reply
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response)

# response["structured_response"] is a CapitalInfo Python object
# (not a string!). Printing it shows the Pydantic model representation.
response["structured_response"]

{'messages': [HumanMessage(content='What is the capital of The Moon?', additional_kwargs={}, response_metadata={}, id='680fc034-3b0d-4b4d-a056-b57375ce9b22'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'cn5t7532t', 'function': {'arguments': '{"economy":"Interplanetary Trade","location":"The Moon","name":"Lunar Haven","vibe":"Futuristic"}', 'name': 'CapitalInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 266, 'total_tokens': 306, 'completion_time': 0.079591044, 'completion_tokens_details': None, 'prompt_time': 0.026117348, 'prompt_tokens_details': None, 'queue_time': 0.16075121, 'total_time': 0.105708392}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2c9f-388d-7b31-a360-530694eb6834-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'economy': 'Interplanetar

CapitalInfo(name='Lunar Haven', location='The Moon', vibe='Futuristic', economy='Interplanetary Trade')

In [13]:
# ============================================================
# CELL 7: Access Individual Fields by Name
# ============================================================
# Because structured_response is a Pydantic object (not a dict
# or a string), we access fields using dot notation — exactly
# the same as any Python class attribute.
#
# This is the key benefit over structured prompts:
#   Structured prompt: response['messages'][1].content → string
#                      then you must parse "Name: Selenopolis\n..."
#
#   Structured output: response['structured_response'].name → "Selenopolis"
#                      clean, typed, no parsing needed
#
# This makes the output trivially easy to pass into:
#   - a database (insert name=capital_info.name)
#   - a downstream API call
#   - a Jinja2 template for rendering HTML
#   - any other part of your pipeline

response["structured_response"].name   # Just the city name, as a plain Python string

'Lunar Haven'

In [14]:
# ============================================================
# CELL 8: Use Structured Fields in Code
# ============================================================
# This cell shows the real-world payoff of structured output:
# we can extract multiple fields and use them in an f-string
# (or any other code) without any string manipulation.
#
# Pattern to remember:
#   capital_info = response["structured_response"]   # get the object
#   capital_info.name                                 # field access
#   capital_info.location                             # another field
#   capital_info.vibe
#   capital_info.economy
#
# In a real application you might do:
#   db.insert(name=capital_info.name, location=capital_info.location, ...)
#   or send capital_info.model_dump() as a JSON payload to an API

capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

# Clean, readable output — no string parsing required
print(f"{capital_name} is a city located at {capital_location}")

Lunar Haven is a city located at The Moon


---
## Summary — The Prompting Ladder

| Technique | How to enable it | Best for |
|---|---|---|
| **Basic prompt** | Pass a string or HumanMessage | Exploration, one-off questions |
| **System prompt** | `system_prompt=` in `create_agent` | Setting persona, rules, task scope |
| **Few-shot examples** | Include Q&A pairs in the system prompt | Enforcing output style or naming conventions |
| **Structured prompt** | Describe a text template in the system prompt | Human-readable formatted output |
| **Structured output** | `response_format=YourPydanticClass` | Machine-readable output; feeding data into code |

### Golden rules

1. **Be specific** — vague instructions produce inconsistent results
2. **Show, don't just tell** — few-shot examples beat long explanations
3. **Use structured output in production** — parsing free text is fragile
4. **Iterate** — prompt engineering is experimental; test with varied inputs

### Next steps
- **1.2** — Adding Tools to your Agent: web search, calculators, database queries
- **1.3** — Memory and multi-turn conversations: how to persist context across sessions